# LongFlow — split-path polish probe (+ stutter fixes)

Runtime: **L4 or A100**, ~45–60 min, ~$3. No training. Pre-registration:
NOTES "SPLIT-PATH POLISH" note under the polish-night results (2026-08-18).

Last night's loop-polish used ONE polished latent for both the audio and
the feedback. This probe separates the channels on the cleanabl base:

| tag | audio path | feedback path |
|---|---|---|
| `sp_audio_k3_s0` / `_s1` | polished k=3 | raw flow latent |
| `sp_feed_k2_s0` | raw flow latent | polished k=2 |
| `sp_both_k2_ema_s0` | polished k=2 | polished k=2, EMA-correlated noise |

Predictions on record: audio-only keeps plain-cleanabl content (WER ~0.12)
with k3 texture — the best-sounding config available today if true;
feedback-only shows whether the identity lift (0.474→0.522) came from the
loop side. The EMA-noise arm + the crossfaded knob re-render test the two
stutter suspects.

| cell | what |
|---|---|
| 1 | cold start (no caches needed beyond held-out utts + ckpt) |
| 2 | polish helpers (EMA noise) + crossfaded decode + knob k3 re-render |
| 3 | closed loop: 4 renders |
| 4 | bundle → Drive root `splitpath_eval.zip` |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "Split-path polish v1.0 (2026-08-18)"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
from pathlib import Path
import numpy as np
import soundfile as sf

if os.path.exists("/content/LongFlow/src"):
    !cd /content/LongFlow && git pull -q
else:
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone failed — check repo access"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.cache.capture import load_utterance
from src.flow_head.cfm import heun_sample
from src.flow_head.integration import CFGFlowHeadPatch, _CFGField
from src.flow_head.trainer import load_checkpoint

CACHE_V2_DRIVE = "/content/drive/MyDrive/longflow_p1_cache_v2"
CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_V1 = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
GATE3_DIR = "/content/drive/MyDrive/longflow_gate3"
OUT = "/content/splitpath"
DRIVE_OUT = "/content/drive/MyDrive/longflow_splitpath"
for d in (OUT, DRIVE_OUT, f"{DRIVE_OUT}/knobfix"):
    os.makedirs(d, exist_ok=True)

# only the two knob utterances come from the cache — copy just those two files
KNOB_UTTS = ["cv2_150w_016a623d", "cv2_1200w_026ad358"]
LOCAL_UTTS = "/content/knob_utts"
os.makedirs(LOCAL_UTTS, exist_ok=True)
for u in KNOB_UTTS:
    if not os.path.exists(f"{LOCAL_UTTS}/{u}.pt"):
        shutil.copy(f"{CACHE_V2_DRIVE}/{u}.pt", LOCAL_UTTS)
print("knob utterances local")

if os.path.exists(f"{DRIVE_OUT}/splitpath_report.json"):
    with open(f"{DRIVE_OUT}/splitpath_report.json") as f:
        report = json.load(f)
    print(f"resuming: {len(report['runs'])} runs already recorded")
else:
    report = {"notebook_version": NOTEBOOK_VERSION, "runs": []}

def done(tag):
    return os.path.exists(f"{DRIVE_OUT}/{tag}.wav")

def save_wav(tag, wav, meta, sub=""):
    d = f"{DRIVE_OUT}/{sub}" if sub else DRIVE_OUT
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{d}/{tag}.wav")
    report["runs"] = [r for r in report["runs"] if r.get("tag") != tag] + [{"tag": tag, **meta}]
    with open(f"{DRIVE_OUT}/splitpath_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"saved {tag}: {len(wav)/24000:.1f}s  {meta}", flush=True)

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

head_cl, mean_cl, std_cl = load_checkpoint(f"{CKPT_DIR}/cleanabl_20k_step20000.pt")
head_cl = head_cl.to("cuda")
print("READY")


In [ ]:
# ===== Polish helpers (EMA-correlated noise) + CROSSFADED decode + knob k3 re-render =====
class PolishNoise:
    """EMA-correlated noise stream: n_t = rho*n_{t-1} + sqrt(1-rho^2)*fresh.
    rho=0 -> independent per call (last night's behavior); rho~0.9 ->
    temporally smooth polish direction (stutter-fix candidate). Never a
    FIXED vector (the seeded-x0 under-dispersion trap, review-adversarial
    2c)."""

    def __init__(self, rho=0.9):
        self.rho = rho
        self.prev = None

    def __call__(self, shape, device, dtype):
        fresh = torch.randn(shape, device=device, dtype=torch.float32)
        if self.prev is None or self.prev.shape != fresh.shape:
            self.prev = fresh
        else:
            self.prev = self.rho * self.prev + (1 - self.rho ** 2) ** 0.5 * fresh
        return self.prev.to(dtype)

def polish(z, cond, neg, k, cfg_scale=1.3, total_steps=10, noise_fn=None):
    if k <= 0:
        return z
    sched = model.model.noise_scheduler
    sched.set_timesteps(total_steps)
    ts = sched.timesteps[-k:]
    zt = z.to("cuda", torch.bfloat16)
    cond2 = torch.cat([cond, neg], dim=0).to("cuda", torch.bfloat16)
    if noise_fn is None:
        noise = torch.randn(zt.shape, device="cuda", dtype=torch.float32).to(torch.bfloat16)
    else:
        noise = noise_fn(zt.shape, "cuda", torch.bfloat16)
    zt = sched.add_noise(zt, noise, ts[0].expand(zt.shape[0]))
    for t in ts:
        combined = torch.cat([zt, zt], dim=0)
        eps = model.model.prediction_head(
            combined, t.repeat(combined.shape[0]).to(combined), condition=cond2)
        c_eps, u_eps = torch.split(eps, len(eps) // 2, dim=0)
        guided = u_eps + cfg_scale * (c_eps - u_eps)
        zt = sched.step(guided, t, zt).prev_sample
    return zt.float()

def decode_latents_xfade(z, chunk_frames=225, overlap_frames=4):
    """Chunked decode with overlapping windows + linear crossfade (~0.5 s at
    7.5 Hz x 4 frames) — kills the 30s butt-joint seam suspect."""
    sc = model.model.speech_scaling_factor
    bi = model.model.speech_bias_factor
    z = z.to("cuda", torch.bfloat16)
    z = z / sc - bi
    step = chunk_frames - overlap_frames
    pieces, shape_fn = [], None
    for i in range(0, z.shape[0], step):
        chunk = z[i : i + chunk_frames]
        candidates = [shape_fn] if shape_fn is not None else [
            lambda c: c.unsqueeze(0), lambda c: c.unsqueeze(0).transpose(1, 2)
        ]
        decoded = None
        for fn in candidates:
            try:
                out = model.model.acoustic_tokenizer.decode(fn(chunk))
                decoded = out[0] if isinstance(out, tuple) else out
                shape_fn = fn
                break
            except Exception as e:
                print(f"decode attempt failed: {repr(e)[:120]}")
        if decoded is None:
            raise RuntimeError("both decode shapes failed")
        pieces.append(decoded.detach().float().cpu().numpy().squeeze())
        del out, decoded
        torch.cuda.empty_cache()
        if i + chunk_frames >= z.shape[0]:
            break
    if len(pieces) == 1:
        return pieces[0]
    spf = len(pieces[0]) // chunk_frames  # audio samples per frame
    ov = overlap_frames * spf
    outw = pieces[0]
    for p in pieces[1:]:
        fade = np.linspace(0, 1, ov, dtype=np.float32)
        outw[-ov:] = outw[-ov:] * (1 - fade) + p[:ov] * fade
        outw = np.concatenate([outw, p[ov:]])
    return outw

# --- knob k3 re-render with both fixes, teacher-forced, A/B vs last night ---
for u in KNOB_UTTS:
    tag = f"{u}_k3_fixed"
    wpath = f"{DRIVE_OUT}/knobfix/{tag}.wav"
    if os.path.exists(wpath):
        print(f"{tag} already on Drive")
        continue
    utt = load_utterance(f"{LOCAL_UTTS}/{u}.pt")
    field = _CFGField(head_cl, utt.neg_hidden.float().cuda(), 1.3)
    g = torch.Generator(device="cuda").manual_seed(0)
    z = heun_sample(field, utt.hidden.float().cuda(), head_cl.cfg.d_latent,
                    nfe=8, sway=0.0, generator=g)
    z = z * std_cl.cuda() + mean_cl.cuda()
    zk = polish(z, utt.hidden.float().cuda(), utt.neg_hidden.float().cuda(),
                k=3, noise_fn=PolishNoise(rho=0.9))
    wav = decode_latents_xfade(zk)
    sf.write(wpath, wav, 24000)
    print(f"{tag}: rendered with EMA noise + crossfade decode — A/B vs last night's k3", flush=True)
print("knobfix wavs in Drive/longflow_splitpath/knobfix/")


In [ ]:
# ===== Closed loop: split-path arms (cleanabl base, GN5-8 protocol) =====
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} — retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern}")

V1_LOCAL = "/content/v1texts"
if len(glob.glob(f"{V1_LOCAL}/*.pt")) < 300:
    os.makedirs(V1_LOCAL, exist_ok=True)
    for f in drive_glob(f"{TRAIN_CACHE_V1}/*.pt")[-300:]:
        shutil.copy(f, V1_LOCAL)
sents = []
for f in sorted(glob.glob(f"{V1_LOCAL}/*.pt")):
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
P0 = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")[0]

def turnscript(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return "\n".join(turns) + "\n"

ABL_WORDS, w = [], 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800:
        break
ABL_SCRIPT = turnscript(ABL_WORDS)
print("closed-loop script:", w, "words")
report["cl_script"] = ABL_SCRIPT
report["cl_words"] = w

class SplitPolishPatch(CFGFlowHeadPatch):
    """Separate polish settings for the audio path (what the recorded latent/
    waveform sees) and the feedback path (what re-enters the LM). VibeVoice
    consumes our return value for BOTH — so this patch returns the FEEDBACK
    latent and stores the AUDIO latent per frame; the audio waveform is
    re-decoded from stored latents after generation."""

    def __init__(self, *args, audio_k=0, feed_k=0, audio_rho=0.0, feed_rho=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.audio_k = audio_k
        self.feed_k = feed_k
        # RULE (probe finding, 2026-08-18): the feedback path may only ever
        # receive INDEPENDENT noise — correlated noise in the loop is the
        # GN5 poison; temporal smoothing is legal on the audio path alone.
        self.audio_noise = PolishNoise(audio_rho) if audio_rho > 0 else None
        self.feed_noise = PolishNoise(feed_rho) if feed_rho > 0 else None
        self.audio_latents = []

    def __enter__(self):
        super().__enter__()
        inner = self.model.sample_speech_tokens
        patch = self

        def flow_sample_split(condition, neg_condition=None, cfg_scale=None):
            z = inner(condition, neg_condition=neg_condition, cfg_scale=cfg_scale)
            zf = z.float()
            c = condition.float().cuda()
            ng = neg_condition.float().cuda() if neg_condition is not None else None
            z_audio = (polish(zf, c, ng, patch.audio_k, noise_fn=patch.audio_noise)
                       if ng is not None and patch.audio_k > 0 else zf)
            z_feed = (polish(zf, c, ng, patch.feed_k, noise_fn=patch.feed_noise)
                      if ng is not None and patch.feed_k > 0 else zf)
            patch.audio_latents.append(z_audio.detach().float().cpu())
            return z_feed.to(condition.dtype)

        self.model.sample_speech_tokens = flow_sample_split
        return self

CL_ARMS = [  # (tag, seed, audio_k, feed_k, audio_rho, feed_rho)
    ("sp_audio_k3_s0", 0, 3, 0, 0.9, 0.0),
    ("sp_audio_k3_s1", 1, 3, 0, 0.9, 0.0),
    ("sp_feed_k2_s0", 0, 0, 2, 0.0, 0.9),
    ("sp_both_k2_ema_s0", 0, 2, 2, 0.9, 0.9),
    # THE STACK (added after the probe's attribution read): audio k3 +
    # feedback k2 with INDEPENDENT noise — last night's identity-lift
    # recipe on the loop side, tonight's content-record recipe on the
    # audio side. Bars: WER <= 0.06 AND sim_med >= 0.50, both seeds.
    ("sp_stack_s0", 0, 3, 2, 0.9, 0.0),
    ("sp_stack_s1", 1, 3, 2, 0.9, 0.0),
]
for tag, seed, ak, fk, arho, frho in CL_ARMS:
    if done(tag):
        print(f"{tag}: already on Drive — skipping")
        continue
    torch.manual_seed(seed)
    with SplitPolishPatch(model, head_cl, mean_cl, std_cl, nfe=8, sway=0.0,
                          sampler=heun_sample, audio_k=ak, feed_k=fk,
                          audio_rho=arho, feed_rho=frho) as patch, torch.inference_mode():
        gen = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=3000)
    # audio path: re-decode from the stored (possibly differently-polished) latents
    if patch.audio_k != patch.feed_k:
        za = torch.cat(patch.audio_latents)
        wav = decode_latents_xfade(za)
    else:
        wav = gen.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    zs = torch.cat(patch.audio_latents) if patch.audio_latents else torch.zeros(1)
    save_wav(tag, wav, {"seed": seed, "audio_k": ak, "feed_k": fk,
                        "audio_rho": arho, "feed_rho": frho,
                        "frames": patch.calls,
                        "latent_std": round(float(zs.std()), 3)})


In [ ]:
# ===== Bundle -> Drive root =====
import zipfile
teacher_ref = f"{GATE3_DIR}/t1_turnsplit_p0.wav"
assert os.path.exists(teacher_ref), "GN3 teacher reference missing from Drive"
with open(f"{DRIVE_OUT}/splitpath_report.json", "w") as f:
    json.dump(report, f, indent=2)

ZIP = "/content/drive/MyDrive/splitpath_eval.zip"
with zipfile.ZipFile(ZIP, "w") as z:
    z.write(teacher_ref, "t1_turnsplit_p0.wav")
    z.write(f"{DRIVE_OUT}/splitpath_report.json", "splitpath_report.json")
    for f in os.listdir(DRIVE_OUT):
        if f.endswith(".wav"):
            z.write(f"{DRIVE_OUT}/{f}", f"closed_loop/{f}")
print(f"bundle at {ZIP} ({os.path.getsize(ZIP)/1e6:.0f} MB) — run score_splitpath_gpu_colab.ipynb next")
